# 02. 화학 데이터 전처리 및 분자 특성 생성

강의 1에서 모은 화학 데이터를 **머신러닝에 쓸 수 있는 형태**로 다듬음


## 0. 환경 설정

- 실습에 필요한 라이브러리를 불러옴 (Google Colab에는 모두 기본 설치되어 있음)

| 라이브러리 | 역할 |
|---|---|
| `pandas` | 표(DataFrame) 형태로 데이터 다룸 |
| `numpy` | 숫자·배열 계산 |
| `matplotlib` | 그래프 시각화 |
| `rdkit.Chem` | 분자를 다루는 핵심 도구 |
| `Descriptors`, `Draw` | 물성 계산 / 분자 그림 |
| `rdMolStandardize` | 분자 표준화(정리) |
| `rdFingerprintGenerator` | 지문(Fingerprint) 생성 |
| `DataStructs` | 지문 ↔ 배열 변환 |


In [ ]:
# [1] Colab에서 RDKit 설치 (로컬에 이미 설치되어 있으면 이 줄은 생략 가능)
# !pip install rdkit

# [2] 필요한 라이브러리 불러오기
import pandas as pd                                  # 표(데이터프레임) 다루기
import numpy as np                                   # 숫자/배열 계산
import matplotlib.pyplot as plt

from rdkit import Chem                               # 분자를 다루는 핵심 도구
from rdkit.Chem import Descriptors, Draw             # 물성 계산, 분자 그림 그리기
from rdkit.Chem.MolStandardize import rdMolStandardize  # 분자 표준화(정리) 도구
from rdkit.Chem import rdFingerprintGenerator        # 지문(Fingerprint) 생성 도구
from rdkit import DataStructs


## 1. SMILES 소개

- **SMILES**: 분자 구조를 **한 줄 텍스트**로 표현하는 방법
- 예) 아스피린 → `CC(=O)Oc1ccccc1C(=O)O`

> 🖼️ **[그림 자리]** — 아스피린 구조식과 SMILES 문자열 대응

**문제점**
- 같은 분자라도 **여러 방식으로 표기 가능** (한 사람을 여러 이름으로 부르는 것과 비슷함)

**해결책 — 정규화(Canonical, 캐노니컬)**
- 하나의 분자를 항상 **똑같은 형태로 통일**함
- 같은 분자는 항상 같은 SMILES가 되어, 중복 확인·비교가 쉬워짐


In [ ]:
# [1] '아스피린'을 서로 다르게 쓴 SMILES 여러 개
#     같은 분자라도 원자 순서, 방향고리 표기(소문자 c / 대문자 KEKULE),
#     작용기 표기 순서 등을 바꾸면 겉모습이 전혀 다른 문자열이 됩니다.
aspirin_smiles = [
    'CC(=O)Oc1ccccc1C(=O)O',
    'O=C(O)c1ccccc1OC(C)=O',
    'c1ccc(C(=O)O)c(OC(C)=O)c1'
]

In [ ]:
# [2] 각 SMILES를 분자로 바꾸고, 정규화(Canonical) SMILES로 통일해 보기
# - SMILES 1 > CC(=O)Oc1ccccc1C(=O)O
smiles_1 = 'CC(=O)Oc1ccccc1C(=O)O'   # aspirin_smiles[0]

# 문자 SMILES(smiles_1)이용 RDKit 분자(mol_1) 생성
mol_1 = Chem.MolFromSmiles(smiles_1)

# 생성된 분자(mol_1) 시각화
display(Draw.MolToImage(mol_1, size=(400, 300)))

# 문자 SMILES(smiles_1)의 정규화 SMILES(canonical_1) 생성
canonical_1 = Chem.MolToSmiles(mol_1, canonical=True)

# 문자 SMILES(smiles_1) & 정규화 SMILES(canonical_1) 출력 및 비교
print("SMILES 1:", smiles_1)
print("Canonical 1:", canonical_1)

In [ ]:
# - SMILES 2 > O=C(O)c1ccccc1OC(C)=O
smiles_2 = 'O=C(O)c1ccccc1OC(C)=O'   # aspirin_smiles[1]

# 문자 SMILES(smiles_2)이용 RDKit 분자(mol_2) 생성
mol_2 = Chem.MolFromSmiles(smiles_2)

# 생성된 분자(mol_2) 시각화
display(Draw.MolToImage(mol_2, size=(400, 300)))

# 문자 SMILES(smiles_2)의 정규화 SMILES(canonical_2) 생성
canonical_2 = Chem.MolToSmiles(mol_2, canonical=True)

# 문자 SMILES(smiles_2) & 정규화 SMILES(canonical_2) 출력 및 비교
print("SMILES 2:", smiles_2)
print("Canonical 2:", canonical_2)

In [ ]:
# - SMILES 3 > c1ccc(C(=O)O)c(OC(C)=O)c1
smiles_3 = 'c1ccc(C(=O)O)c(OC(C)=O)c1'   # aspirin_smiles[2]

# 문자 SMILES(smiles_3)이용 RDKit 분자(mol_3) 생성
mol_3 = Chem.MolFromSmiles(smiles_3)

# 생성된 분자(mol_3) 시각화
display(Draw.MolToImage(mol_3, size=(400, 300)))

# 문자 SMILES(smiles_3)의 정규화 SMILES(canonical_3) 생성
canonical_3 = Chem.MolToSmiles(mol_3, canonical=True)

# 문자 SMILES(smiles_3) & 정규화 SMILES(canonical_3) 출력 및 비교
print("SMILES 3:", smiles_3)
print("Canonical 3:", canonical_3)

## 2. 분자 특성 및 분자 구조

### 분자 특성 (Descriptor)
- 분자의 성질을 **숫자**로 나타낸 것을 Descriptor(디스크립터, 분자 물성)라 함
- 머신러닝 모델은 숫자를 입력으로 받으므로, 분자를 숫자로 바꿔줘야 함
- 여기서는 대표적인 6가지를 계산함

| 이름 | 의미 |
|---|---|
| MolWt | 분자량 |
| MolLogP | 기름/물에 대한 친화도 |
| TPSA | 극성 표면적 |
| NumHDonors | 수소 공여자 개수 |
| NumHAcceptors | 수소 수용자 개수 |
| NumRotatableBonds | 회전 가능한 결합 수 |

### 분자 구조 Fingerprint
- 분자 구조를 **0/1 비트 배열**로 나타낸 것이 Fingerprint임
- Morgan(ECFP)은 각 원자 주변 환경(반경)을 훑어 비트로 기록함

> 🖼️ **[그림 자리]** — Morgan Fingerprint 생성 원리 (원자 → 반경 확장 → 해싱 → 비트 배열)


In [ ]:
# 아스피린의 canonical SMILES
aspirin_smiles = 'CC(=O)Oc1ccccc1C(=O)O'

# canoncial SMILES을 활용한 RDKit 분자 생성
mol = Chem.MolFromSmiles(aspirin_smiles)

In [ ]:
# 분자 특성(Descriptor)
# 분자량(MolWt) / 지용성(LogP) / 극성표면적(TPSA)
# NumHDonors / NumHAcceptors / NumRotatableBonds

mol_wt = Descriptors.MolWt(mol)
logp = Descriptors.MolLogP(mol)
tpsa = Descriptors.TPSA(mol)
num_h_donor = Descriptors.NumHDonors(mol)
num_h_acceptor = Descriptors.NumHAcceptors(mol)
num_rotatable = Descriptors.NumRotatableBonds(mol)

print("MolWt             :", mol_wt)
print("LogP              :", logp)
print("TPSA              :", tpsa)
print("NumHDonor         :", num_h_donor)
print("NumHAcceptor      :", num_h_acceptor)
print("NumRotatableBonds :", num_rotatable)

In [ ]:
# FingerPrint(MorganFP, r=2, bit=2048)
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

# Morgan fingerprint 생성
morgan_fp = morgan_generator.GetFingerprint(mol)

print("\nMorgan Fingerprint")
print("-" * 30)
print("Fingerprint length :", morgan_fp.GetNumBits())
print("Number of on-bits  :", morgan_fp.GetNumOnBits())
print("On-bit indices     :", list(morgan_fp.GetOnBits()))

In [ ]:
# 생성된 Fp의 Bit 시각적으로 확인
# 2048 bit --> 32 X 64 의 matrix로 차원 변환하여 확인

# RDKit ExplicitBitVect -> numpy array
fp_array = np.zeros((2048,), dtype=int)
DataStructs.ConvertToNumpyArray(morgan_fp, fp_array)

# 32 x 64로 reshape
fp_img = fp_array.reshape(32, 64)

# 32 X 64 matrix 시각화 (matrix의 좌표의 값에 따라 0 = 흰색 / 1 = 검은색)
plt.figure(figsize=(12, 4))
plt.imshow(fp_img, cmap='binary', aspect='auto')
plt.title("Morgan Fingerprint (2048 bits)")
plt.xlabel("Bit position (grouped)")
plt.ylabel("Rows")
plt.show()

## 3. [실습] 다양한 Molecular Fingerprint 비교

> **직접 해보기 🖐️**
>
> - 앞에서는 **Morgan(ECFP)** 지문을 만들어 32×64 이미지로 그려봄
> - 이번에는 **Morgan이 아닌 다른 지문**을 하나 골라 똑같이 시각화함
> - Morgan과 켜진 비트(on-bit) 개수가 얼마나 다른지 비교해 봄

💡 **RDKit의 대표적인 Fingerprint 종류**

| 종류 | 만드는 방법 | 특징 |
|---|---|---|
| Morgan (ECFP) | `GetMorganGenerator(radius=2, fpSize=2048)` | 원자 주변 환경(반경) 기반, 가장 널리 쓰임 |
| RDKit (Topological) | `GetRDKitFPGenerator(fpSize=2048)` | 분자 내 경로(path) 기반 |
| AtomPair | `GetAtomPairGenerator(fpSize=2048)` | 원자쌍 + 거리 정보 기반 |
| TopologicalTorsion | `GetTopologicalTorsionGenerator(fpSize=2048)` | 연속된 4원자(비틀림) 기반 |
| MACCS keys | `MACCSkeys.GenMACCSKeys(mol)` | 미리 정해진 167개 부분구조 |

- 위 4가지 생성기는 모두 `generator.GetFingerprint(mol)` 로 지문을 만듦
- 아래 빈칸 `______` 을 채워서 완성해 봄


In [ ]:
# [1] Morgan이 아닌 다른 Fingerprint 생성기 하나 고르기
#     힌트: 위 표에서 골라 빈칸을 채우세요. (예: GetRDKitFPGenerator / GetAtomPairGenerator / GetTopologicalTorsionGenerator)
other_generator = rdFingerprintGenerator.______(fpSize=2048)

# [2] 아스피린 분자(mol)로 지문 만들기
#     힌트: 어떤 생성기든 generator.GetFingerprint(mol) 형태로 사용합니다.
other_fp = other_generator.______(mol)

# [3] 켜진 비트(on-bit) 개수를 Morgan과 비교
print("Morgan   on-bits :", morgan_fp.GetNumOnBits())   # 앞 셀에서 만든 Morgan 지문
print("New FP   on-bits :", other_fp.GetNumOnBits())

# [4] RDKit 지문 -> numpy 배열로 변환
fp_array = np.zeros((2048,), dtype=int)
DataStructs.ConvertToNumpyArray(other_fp, fp_array)

# [5] Morgan 때와 똑같이 32 x 64로 reshape 해서 이미지로 그리기
fp_img = fp_array.reshape(32, 64)

plt.figure(figsize=(12, 4))
plt.imshow(fp_img, cmap='binary', aspect='auto')
plt.title("Other Fingerprint (2048 bits)")
plt.xlabel("Bit position (grouped)")
plt.ylabel("Rows")
plt.show()

## 4. 데이터 전처리

- 원시 데이터를 아래 순서로 정리함

| 순서 | 단계 | 목적 |
|---|---|---|
| 1 | 데이터 로드 | CSV 불러오기 |
| 2 | 중복 제거 | 같은 분자 하나만 남김 |
| 3 | SMILES 유효성 검사 | 읽을 수 없는 구조 제거 |
| 4 | 염/무기물 제거 | 주 분자만 남김 |
| 5 | SMILES 정규화 | Canonical 형태로 통일 |

> 🖼️ **[그림 자리]** — 전처리 5단계 흐름도 (건수 변화 포함)


### 4-1. 데이터 로드

- 강의 1에서 저장한 CSV 파일을 불러옴


In [ ]:
# [1] 강의 1에서 만든 데이터 파일을 불러오기
df = pd.read_csv('hif2a_aid651580_dataset.csv')

# [2] 데이터의 크기 확인 (행 개수, 열 개수)
print('데이터 크기 (행, 열):', df.shape)

# [3] 맨 앞 5줄을 화면에 보여주기
df.head()

### 4-2. 중복 제거

- 같은 SMILES(같은 분자)가 여러 번 들어있을 수 있음
- 중복은 하나만 남김


In [ ]:
# [1] 중복 제거 전 행 개수 기록
before = len(df)

# [2] SMILES 기준으로 중복된 분자는 하나만 남기기
df = df.drop_duplicates(subset=['PUBCHEM_SMILES'])

# [3] 중복 제거 후 행 개수
after = len(df)

# [4] 몇 개가 줄었는지 출력
print('중복 제거 전:', before, '개')
print('중복 제거 후:', after, '개')
print('제거된 중복:', before - after, '개')

### 4-3. SMILES 유효성 검사

- 가끔 잘못 적힌 SMILES가 섞여 있음
- `Chem.MolFromSmiles()`로 읽어봄
  - 결과가 `None` → **읽을 수 없는(잘못된) 구조** → 버림


In [ ]:
# [1] 각 SMILES가 올바른지 True/False 로 표시하는 리스트 만들기
#     MolFromSmiles 결과가 None 이 아니면 올바른 분자

# 유효성 검사 결과를 저장할 빈 공간(리스트)
is_valid = []

# 데이터내의 행(분자) 개수
mol_count = len(df)

# 첫 번째 부터 마지막 분자까지 하나씩 검사
for mol_index in range(mol_count):

    # mol_index 번째 분자의 SMILES 가져오기
    smiles = df.iloc[mol_index]['PUBCHEM_SMILES']

    # SMILES가 있는지 없는지 검사
    # - 없는 경우 유효성 검사 진행하지 않고 False 결과 부여
    if pd.isna(smiles):
        is_valid.append(False)

    # - 있는 경우 유효성 검사 실시
    else:
        # smiles를 RDKit 분자 객체로 변환
        mol = Chem.MolFromSmiles(smiles)

        # 분자 변환 성공 여부 확인
        # - 성공 : 유효성 검사 True
        if mol is not None:
            is_valid.append(True)
        # - 실패 : 유효성 검사 False
        else:
            is_valid.append(False)

# [2] 검사 전 행 개수 기록
before = len(df)

# [3] 올바른(True) 분자만 남기기
# #   --> is_valid 공간내에 True라고 표시되는 분자 순서만 남김
df = df[is_valid]

# [4] 몇 개가 제거되었는지 출력
print('유효성 검사 전:', before, '개')
print('유효성 검사 후:', len(df), '개')
print('제거된(잘못된) 구조:', before - len(df), '개')

### 4-4. 염/무기물 제거 & 표준화 (Canonical SMILES)

- 약물 데이터에는 분자에 **염(salt)** 이나 작은 이온이 붙어있는 경우가 많음
- 처리 방침
  - **가장 큰 조각(주 분자)** 만 남김
  - 전하를 중성으로 정리함
  - **정규화된 SMILES(Canonical SMILES)** 로 통일함

**`standardize` 함수가 하는 일**

| 단계 | 메서드 | 기능 |
|---|---|---|
| 1 | `rdMolStandardize.Cleanup(mol)` | 원자·결합 표현을 표준 형태로 정리 |
| 2 | `rdMolStandardize.FragmentParent(mol)` | 여러 조각 중 주 분자만 남김 (예: Na⁺, Cl⁻ 등 염 제거) |
| 3 | `Uncharger().uncharge(mol)` | 전하를 가능한 한 중성으로 정리 |
| 4 | 허용 원소 필터 | 허용 목록 밖 원소(금속 등) 포함 분자 제거 |
| 5 | `Chem.MolToSmiles(canonical=True)` | 정규화 SMILES 문자열로 변환 |




In [ ]:
# [1] 정규화된 SMILES를 저장할 빈 공간(리스트) 만들기
standardized_smiles = []

# [2] 전하를 중성 형태로 정리하는 도구 만들기
uncharger = rdMolStandardize.Uncharger()

# [3] 일반적인 유기분자에서 허용할 원소 지정
allowed_elements = [
    'H', 'B', 'C', 'N', 'O',
    'F', 'P', 'S', 'Cl', 'Br', 'I'
]


# [4] 데이터프레임의 SMILES를 한 행씩 가져오기
for row_index in range(len(df)):

    # 현재 행의 SMILES 가져오기
    smi = df.iloc[row_index]['PUBCHEM_SMILES']

    # 텍스트 SMILES를 RDKit 분자 객체로 변환
    mol = Chem.MolFromSmiles(smi)

    # 원자와 결합 표현을 기본적인 표준 형태로 정리
    mol = rdMolStandardize.Cleanup(mol)

    # 여러 조각으로 이루어진 경우 주성분만 남기기
    # 예: 약물 분자와 함께 포함된 Na+, Cl- 등의 염 제거
    mol = rdMolStandardize.FragmentParent(mol)

    # 분자에 표시된 전하를 가능한 경우 중성 형태로 정리
    mol = uncharger.uncharge(mol)


    # -----------------------------------------------------
    # 허용되지 않은 원소가 포함되어 있는지 확인
    # -----------------------------------------------------

    # 처음에는 허용되지 않은 원소가 없다고 가정
    has_disallowed_element = False

    # 분자에 포함된 원자를 하나씩 확인
    for atom in mol.GetAtoms():

        # 현재 원자의 원소 기호 가져오기
        element = atom.GetSymbol()

        # 허용 원소 목록에 없는 원소가 발견된 경우
        if element not in allowed_elements:
            has_disallowed_element = True


    # 금속 등 허용되지 않은 원소가 포함된 분자는 제거 대상으로 표시
    if has_disallowed_element == True:
        standardized_smiles.append(None)
        continue


    # 정리된 분자를 Canonical SMILES 문자열로 변환
    canonical_smi = Chem.MolToSmiles(
        mol,
        canonical=True
    )

    # 변환된 Canonical SMILES를 리스트에 저장
    standardized_smiles.append(canonical_smi)


# [5] 표준화된 SMILES 리스트를 새로운 컬럼으로 추가
df['Canonical_SMILES'] = standardized_smiles


# [6] 허용되지 않은 원소가 포함된 행 제거
before = len(df)

df = df.dropna(
    subset=['Canonical_SMILES']
).copy()


# [7] 처리 결과 확인
print('처리 전 데이터 수:', before, '개')
print('처리 후 데이터 수:', len(df), '개')
print('제거된 구조 수:', before - len(df), '개')


# [8] 원본과 정규화된 SMILES 비교
df[
    ['PUBCHEM_SMILES', 'Canonical_SMILES']
].head()

## 5. Descriptor(분자 물성) 및 Fingerprint 생성

### 5-1. 분자 물성 생성 및 정리
- 앞서 아스피린 하나로 해본 계산을, 데이터의 **모든 분자에 대해 반복함**

In [ ]:
# 위에서 아스피린 분자로 해본 계산을, 데이터의 모든 분자에 대해 반복합니다.

# [1] 각 물성을 담을 빈 리스트들 준비
molwt_list, logp_list, tpsa_list = [], [], []
hdonor_list, hacceptor_list, rotbond_list = [], [], []

# [2] Canonical_SMILES 를 하나씩 꺼내서 물성 계산
for smi in df['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)               # 분자 객체로 변환
    molwt_list.append(Descriptors.MolWt(mol))            # 분자량
    logp_list.append(Descriptors.MolLogP(mol))           # LogP
    tpsa_list.append(Descriptors.TPSA(mol))              # 극성 표면적
    hdonor_list.append(Descriptors.NumHDonors(mol))      # 수소 공여자 수
    hacceptor_list.append(Descriptors.NumHAcceptors(mol))# 수소 수용자 수
    rotbond_list.append(Descriptors.NumRotatableBonds(mol))  # 회전 가능 결합 수

# [3] 계산한 값을 데이터프레임의 새 컬럼으로 추가
df['MolWt'] = molwt_list
df['MolLogP'] = logp_list
df['TPSA'] = tpsa_list
df['NumHDonors'] = hdonor_list
df['NumHAcceptors'] = hacceptor_list
df['NumRotatableBonds'] = rotbond_list

# 참고: Descriptors.CalcMolDescriptors(mol) 를 쓰면 약 200개 물성을 한 번에 계산할 수 있습니다.

# [4] 만들어진 물성 표 확인
df[['Canonical_SMILES', 'MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds']].head()

### 5-2. 분자 Fingerprint 생성 및 정리
- 각 분자의 Morgan Fingerprint(2048 bit)를 계산해 **비트별 컬럼**으로 펼침

In [ ]:
# 위에서 아스피린 분자로 해본 계산을
# 데이터의 모든 분자에 대해 반복합니다.
# Morgan Fingerprint의 각 bit를 하나의 column으로 만듭니다.


# [1] Morgan Fingerprint 생성 도구 만들기
# radius = 2, fingerprint 길이 = 2048 bit
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

# [2] 모든 분자의 Fingerprint를 저장할 빈 리스트 만들기
fingerprint_list = []


# [3] Canonical_SMILES를 하나씩 꺼내서 Fingerprint 계산
for smi in df['Canonical_SMILES']:

    # SMILES 문자열을 RDKit 분자 객체로 변환
    mol = Chem.MolFromSmiles(smi)

    # Morgan Fingerprint 계산
    morgan_fp = morgan_generator.GetFingerprint(mol)

    # 2048개의 0으로 이루어진 빈 배열 만들기
    fp_array = np.zeros(
        2048,
        dtype=int
    )

    # RDKit Fingerprint를 NumPy 0/1 배열로 변환
    DataStructs.ConvertToNumpyArray(
        morgan_fp,
        fp_array
    )

    # 현재 분자의 Fingerprint를 리스트에 저장
    fingerprint_list.append(fp_array)


# [4] Fingerprint column 이름 만들기
# FP_0, FP_1, FP_2, ..., FP_2047
fingerprint_columns = []

for bit_index in range(2048):
    column_name = 'FP_' + str(bit_index)
    fingerprint_columns.append(column_name)


# [5] Fingerprint 리스트를 DataFrame으로 변환
fingerprint_df = pd.DataFrame(
    fingerprint_list,
    columns=fingerprint_columns,
    index=df.index
)


# [6] 기존 데이터프레임과 Fingerprint 데이터프레임 합치기
df = pd.concat(
    [df, fingerprint_df],
    axis=1
)


# [7] 결과 확인
print('전체 데이터 크기:', df.shape)
print('Fingerprint 컬럼 수:', len(fingerprint_columns))

df.head()

## 6. 저장

- 전처리한 데이터를 CSV로 저장함
- 이 파일은 **다음 강의(모델 학습)** 에서 사용함


In [ ]:

# [2] CSV 파일로 저장 (index=False: 줄 번호는 저장하지 않음)
df.to_csv('hif2a_preprocessed.csv', index=False)

print('저장 완료: hif2a_preprocessed.csv')
print('저장한 데이터 크기:', df.shape)

# 참고: Fingerprint 는 SMILES 만 있으면 강의 3에서 다시 만들 수 있으므로 CSV에는 저장하지 않습니다.

# [3] (Colab 사용 시) 파일을 내 컴퓨터로 내려받기 - 필요하면 아래 두 줄의 주석을 푸세요
# from google.colab import files
# files.download('hif2a_preprocessed.csv')